# Módulo 2. Tarefa 2. Auditoria de IA: triagem de CVs

Este notebook é a __Parte 4 (ferramentas de IA responsável)__ da tarefa. O sistema auditado é um __sistema fictício de triagem automática de CVs__, de uma empresa imaginária, que decide quem passa à entrevista para vagas técnicas. A ferramenta de auditoria é o __Fairlearn__. A descrição completa do sistema, a avaliação dos princípios, as regulamentações e os impactos estão no PDF de entrega.

__Aviso importante:__ todos os dados deste notebook são __sintéticos__ (criados por nós, com uma semente fixa para o resultado ser sempre o mesmo). O viés foi colocado de propósito nos dados históricos, como no cenário 2 do curso. Por isso, os resultados mostram __como a ferramenta funciona__ e não provam nada sobre nenhuma empresa real.

# Parte 4. Auditoria com o Fairlearn

#### Passo 1: instalar e importar as bibliotecas

O Fairlearn ainda não está instalado. A célula seguinte instala-o (cerca de 140 KB; se já estiver instalado, não faz nada). As outras bibliotecas (NumPy, pandas, matplotlib e scikit-learn) já estão instaladas.

In [ ]:
# Instala o Fairlearn, a biblioteca de auditoria de equidade (fairness)
%pip install fairlearn

In [ ]:
# Importa o NumPy, com o alias habitual np, para gerar números aleatórios
import numpy as np

# Importa o pandas, com o alias habitual pd, para trabalhar com a tabela de candidatos
import pandas as pd

# Importa o pyplot do matplotlib, com o alias habitual plt, para os gráficos
import matplotlib.pyplot as plt

# Importa do scikit-learn: a divisão treino/teste, o modelo de regressão logística e a exatidão
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Importa do Fairlearn: a tabela de métricas por grupo e as três métricas usadas na auditoria
from fairlearn.metrics import MetricFrame, selection_rate, true_positive_rate, false_positive_rate, demographic_parity_difference

# Importa do Fairlearn a técnica de mitigação (ajusta o limiar de decisão de cada grupo)
from fairlearn.postprocessing import ThresholdOptimizer


# Aumenta a largura de impressão das tabelas, para as colunas não serem cortadas em duas linhas
pd.set_option("display.width", 200)


def formatar_percentagem(valor):
    # Converte um número entre 0 e 1 em texto com percentagem e vírgula decimal (ex.: 0.448 -> "44,8%")
    return f"{valor * 100:.1f}".replace(".", ",") + "%"


def formatar_pontos_percentuais(valor):
    # Converte uma diferença entre percentagens em texto (ex.: 0.318 -> "31,8 pontos percentuais")
    return f"{valor * 100:.1f}".replace(".", ",") + " pontos percentuais"


print("Bibliotecas importadas com sucesso.")

#### Passo 2: criar os dados sintéticos

Cria-se uma tabela com __5 000 candidatos fictícios__. Cada candidato tem:

- `genero`: "Feminino" ou "Masculino" (usado __só na auditoria__; o modelo não o vê);
- `anos_experiencia`, `nota_teste_tecnico` (0 a 100) e `nivel_formacao` (1 a 4);
- `anos_lacuna_cv`: anos de interrupção no CV (nestes dados, as mulheres têm lacunas com maior frequência);
- `candidato_apto`: 1 se o candidato é __realmente__ apto para a vaga (só sabemos isto porque os dados são inventados; na vida real não se conhece);
- `chamado_entrevista_historico`: 1 se, no passado, os recrutadores chamaram o candidato à entrevista. Esta é a coluna __enviesada__: os recrutadores penalizavam as lacunas no CV e as mulheres.

A aptidão real depende só da experiência, da nota e da formação, por isso homens e mulheres devem ter taxas de aptidão semelhantes.

In [ ]:
# Cria o gerador de números aleatórios com semente fixa, para o resultado ser sempre o mesmo
gerador_aleatorio = np.random.default_rng(42)
numero_candidatos = 5000

# Gera as características de cada candidato
genero = gerador_aleatorio.choice(["Feminino", "Masculino"], size=numero_candidatos)
anos_experiencia = gerador_aleatorio.integers(0, 16, size=numero_candidatos)
nota_teste_tecnico = np.clip(np.round(gerador_aleatorio.normal(65, 15, size=numero_candidatos)), 0, 100)
nivel_formacao = gerador_aleatorio.integers(1, 5, size=numero_candidatos)

# As lacunas no CV são mais frequentes nas mulheres (este é o padrão que o modelo vai acabar por aprender)
anos_lacuna_cv = np.where(
    genero == "Feminino",
    gerador_aleatorio.choice([0, 1, 2, 3], size=numero_candidatos, p=[0.45, 0.30, 0.18, 0.07]),
    gerador_aleatorio.choice([0, 1, 2, 3], size=numero_candidatos, p=[0.90, 0.06, 0.03, 0.01]),
)

# Pontuação de competência: depende só da experiência, da nota e da formação
pontuacao_competencia = (
    0.15 * anos_experiencia + 0.06 * (nota_teste_tecnico - 65) + 0.4 * (nivel_formacao - 2.5)
)

# Aptidão real do candidato (a "verdade"): competência mais um pouco de ruído
pontuacao_real = pontuacao_competencia + gerador_aleatorio.normal(0, 0.4, size=numero_candidatos)
candidato_apto = (pontuacao_real > 1.3).astype(int)

# Decisão histórica dos recrutadores: a competência, mas penalizando as lacunas no CV e o género feminino
pontuacao_historica = (
    pontuacao_competencia
    + gerador_aleatorio.normal(0, 0.4, size=numero_candidatos)
    - 0.9 * anos_lacuna_cv
    - 0.5 * (genero == "Feminino")
)
chamado_entrevista_historico = (pontuacao_historica > 1.3).astype(int)

# Junta tudo numa tabela (DataFrame) de candidatos
candidatos = pd.DataFrame({
    "genero": genero,
    "anos_experiencia": anos_experiencia,
    "nota_teste_tecnico": nota_teste_tecnico,
    "nivel_formacao": nivel_formacao,
    "anos_lacuna_cv": anos_lacuna_cv,
    "candidato_apto": candidato_apto,
    "chamado_entrevista_historico": chamado_entrevista_historico,
})

print(f"Número de candidatos: {len(candidatos)}")
print("Primeiros 5 candidatos:")
print(candidatos.head())

In [ ]:
# Mostra, para cada género, a percentagem de candidatos realmente aptos e a de chamados à entrevista no passado
print("Dados históricos por género:\n")
for nome_genero in ["Feminino", "Masculino"]:
    candidatos_do_genero = candidatos[candidatos["genero"] == nome_genero]
    taxa_aptos = candidatos_do_genero["candidato_apto"].mean()
    taxa_chamados = candidatos_do_genero["chamado_entrevista_historico"].mean()
    print(f"{nome_genero}: {formatar_percentagem(taxa_aptos)} realmente aptos | "
          f"{formatar_percentagem(taxa_chamados)} chamados à entrevista no passado")

#### Passo 3: treinar o modelo de triagem

O sistema é um __modelo de regressão logística__ do scikit-learn (matéria do Módulo 3). Em termos simples: aprende com exemplos do passado e calcula um __peso__ para cada característica; para um candidato novo devolve uma probabilidade de passar e, acima de 50%, decide "passa à entrevista".

O modelo aprende com os dados __históricos__ (a coluna enviesada) e __não vê o género__. Os dados são divididos em __treino__ (70%, para aprender) e __teste__ (30%, para avaliar com candidatos que o modelo nunca viu).

In [ ]:
# Divide os candidatos em treino (70%) e teste (30%), com semente fixa
candidatos_treino, candidatos_teste = train_test_split(candidatos, test_size=0.3, random_state=42)

# Escolhe as características que o modelo vê (o género NÃO está incluído)
nomes_caracteristicas = ["anos_experiencia", "nota_teste_tecnico", "nivel_formacao", "anos_lacuna_cv"]

# Treina o modelo com a decisão histórica dos recrutadores
modelo_triagem = LogisticRegression(max_iter=1000)
modelo_triagem.fit(candidatos_treino[nomes_caracteristicas], candidatos_treino["chamado_entrevista_historico"])

# Mostra os pesos aprendidos: um peso negativo diminui a probabilidade de o candidato passar
pesos_do_modelo = pd.Series(modelo_triagem.coef_[0], index=nomes_caracteristicas)
print("Pesos aprendidos pelo modelo:")
print(pesos_do_modelo.round(3))

In [ ]:
# Faz a previsão para os candidatos de teste (1 = passa à entrevista, 0 = não passa)
previsoes_modelo = modelo_triagem.predict(candidatos_teste[nomes_caracteristicas])

# Compara as previsões com a decisão histórica (o que o modelo aprendeu) e com a aptidão real
exatidao_contra_historico = accuracy_score(candidatos_teste["chamado_entrevista_historico"], previsoes_modelo)
exatidao_contra_aptidao = accuracy_score(candidatos_teste["candidato_apto"], previsoes_modelo)

print(f"Exatidão contra a decisão histórica: {formatar_percentagem(exatidao_contra_historico)}")
print(f"Exatidão contra a aptidão real: {formatar_percentagem(exatidao_contra_aptidao)}")

#### Passo 4: auditar o modelo com o Fairlearn

O desempenho geral parece bom, mas a auditoria olha para __cada grupo__. O Fairlearn tem a `MetricFrame`, uma tabela que calcula as métricas separadamente por género. Usam-se duas métricas do curso e uma terceira, de apoio:

- __Demographic parity__: a taxa de decisões positivas (candidatos que passam) deve ser igual entre grupos. Mede-se com a `selection_rate` (taxa de seleção).
- __Equal opportunity__: a taxa de verdadeiros positivos deve ser igual entre grupos, isto é, entre os candidatos __realmente aptos__, a percentagem que passa deve ser igual para homens e mulheres.

- __Taxa de falsos positivos__ (de apoio, não é uma das métricas do curso): entre os candidatos __não aptos__, a percentagem que passa por engano. Ajuda a perceber o que acontece quando se corrige uma das outras duas.

Como o curso diz, não existe uma "métrica universal" de justiça algorítmica; por isso olham-se várias.

In [ ]:
# Calcula as métricas separadamente para cada género, comparando as previsões com a aptidão real
metricas_antes_mitigacao = MetricFrame(
    metrics={
        "Taxa de seleção": selection_rate,
        "Taxa de verdadeiros positivos": true_positive_rate,
        "Taxa de falsos positivos": false_positive_rate,
    },
    y_true=candidatos_teste["candidato_apto"],
    y_pred=previsoes_modelo,
    sensitive_features=candidatos_teste["genero"],
)

# Tabela com as métricas por género (valores entre 0 e 1)
print("Métricas por género ANTES da mitigação:")
print(metricas_antes_mitigacao.by_group.round(3))

# Diferença entre os dois grupos (o ideal é 0)
diferencas_antes = metricas_antes_mitigacao.difference()
print(f"\nDiferença na taxa de seleção (demographic parity): {formatar_pontos_percentuais(diferencas_antes['Taxa de seleção'])}")
print(f"Diferença na taxa de verdadeiros positivos (equal opportunity): {formatar_pontos_percentuais(diferencas_antes['Taxa de verdadeiros positivos'])}")

# Confirmação com a função própria do Fairlearn para a demographic parity
diferenca_paridade_demografica_antes = demographic_parity_difference(
    candidatos_teste["candidato_apto"], previsoes_modelo, sensitive_features=candidatos_teste["genero"]
)
print(f"demographic_parity_difference do Fairlearn: {formatar_pontos_percentuais(diferenca_paridade_demografica_antes)}")

In [ ]:
# Gráfico com as duas métricas por género, antes da mitigação
tabela_percentagens_antes = metricas_antes_mitigacao.by_group * 100
tabela_percentagens_antes.plot(kind="bar", rot=0, figsize=(7, 4))
plt.ylim(0, 100)  # escala de 0 a 100% para a legenda não tapar as barras
plt.title("Auditoria do modelo de triagem de CVs (antes da mitigação)")
plt.ylabel("Percentagem (%)")
plt.xlabel("Género")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

#### Passo 5: mitigar o viés

A técnica escolhida é o `ThresholdOptimizer` do Fairlearn, uma mitigação __pós-treino__ (o modelo não é treinado outra vez). Em vez de usar o mesmo limiar de 50% para todos, ajusta o limiar de decisão de __cada grupo__ de forma a igualar a taxa de verdadeiros positivos (equal opportunity).

Para isso precisa de saber a aptidão real dos candidatos de treino, algo que, na vida real, só existe se houver uma forma fiável de saber quem era mesmo apto. Aqui sabemos, porque os dados são sintéticos.

In [ ]:
# Cria o otimizador de limiares, usando o modelo já treinado, com o objetivo de igualar a taxa de verdadeiros positivos
otimizador_limiares = ThresholdOptimizer(
    estimator=modelo_triagem,
    constraints="true_positive_rate_parity",
    prefit=True,
    predict_method="predict_proba",
)

# Ajusta os limiares de cada género com os candidatos de treino e a sua aptidão real
otimizador_limiares.fit(
    candidatos_treino[nomes_caracteristicas],
    candidatos_treino["candidato_apto"],
    sensitive_features=candidatos_treino["genero"],
)

# Faz as novas previsões para os candidatos de teste (semente fixa, porque a técnica tem uma parte aleatória)
previsoes_mitigadas = otimizador_limiares.predict(
    candidatos_teste[nomes_caracteristicas],
    sensitive_features=candidatos_teste["genero"],
    random_state=42,
)

# Repete a auditoria com as novas previsões
metricas_depois_mitigacao = MetricFrame(
    metrics={
        "Taxa de seleção": selection_rate,
        "Taxa de verdadeiros positivos": true_positive_rate,
        "Taxa de falsos positivos": false_positive_rate,
    },
    y_true=candidatos_teste["candidato_apto"],
    y_pred=previsoes_mitigadas,
    sensitive_features=candidatos_teste["genero"],
)

print("Métricas por género DEPOIS da mitigação:")
print(metricas_depois_mitigacao.by_group.round(3))

diferencas_depois = metricas_depois_mitigacao.difference()
print(f"\nDiferença na taxa de seleção (demographic parity): {formatar_pontos_percentuais(diferencas_depois['Taxa de seleção'])}")
print(f"Diferença na taxa de verdadeiros positivos (equal opportunity): {formatar_pontos_percentuais(diferencas_depois['Taxa de verdadeiros positivos'])}")

In [ ]:
# Gráfico de comparação antes e depois da mitigação, para cada métrica
figura, eixos = plt.subplots(1, 3, figsize=(15, 4))

for posicao_eixo, nome_metrica in enumerate(["Taxa de verdadeiros positivos", "Taxa de seleção", "Taxa de falsos positivos"]):
    comparacao_metrica = pd.DataFrame({
        "Antes da mitigação": metricas_antes_mitigacao.by_group[nome_metrica] * 100,
        "Depois da mitigação": metricas_depois_mitigacao.by_group[nome_metrica] * 100,
    })
    comparacao_metrica.plot(kind="bar", rot=0, ax=eixos[posicao_eixo])
    eixos[posicao_eixo].set_ylim(0, 100)  # escala de 0 a 100% para a legenda não tapar as barras
    eixos[posicao_eixo].set_title(nome_metrica)
    eixos[posicao_eixo].set_ylabel("Percentagem (%)")
    eixos[posicao_eixo].set_xlabel("Género")

plt.tight_layout()
plt.show()

#### Resumo dos resultados

A última célula junta os números principais, para a análise do PDF. Interpretação:

- Antes da mitigação, entre os candidatos __realmente aptos__, as mulheres passam bem menos do que os homens, mesmo sem o modelo ver o género: aprendeu o viés através das lacunas no CV.
- A mitigação aproxima as taxas de verdadeiros positivos dos dois géneros. A taxa de seleção __não__ fica igual (e, depois da mitigação, são as mulheres que têm a taxa de seleção mais alta), porque a técnica só procurou igualar a equal opportunity: é o compromisso entre métricas de que fala o curso.
- A taxa de falsos positivos explica porquê: para subir a taxa de verdadeiros positivos das mulheres, o limiar delas baixou muito e passaram também mais mulheres não aptas (os falsos positivos delas subiram bastante).
- Neste exemplo a exatidão contra a aptidão real não desceu com a mitigação. A explicação provável é que o modelo original tinha aprendido com decisões históricas enviesadas; noutros casos, a mitigação pode ter custos.

In [ ]:
# Exatidão contra a aptidão real, antes e depois da mitigação
exatidao_depois_mitigacao = accuracy_score(candidatos_teste["candidato_apto"], previsoes_mitigadas)

print("RESUMO DA AUDITORIA (dados sintéticos, candidatos de teste)\n")
print(f"Candidatos de teste: {len(candidatos_teste)}")
print(f"Diferença na taxa de verdadeiros positivos: {formatar_pontos_percentuais(diferencas_antes['Taxa de verdadeiros positivos'])} antes | "
      f"{formatar_pontos_percentuais(diferencas_depois['Taxa de verdadeiros positivos'])} depois")
print(f"Diferença na taxa de seleção: {formatar_pontos_percentuais(diferencas_antes['Taxa de seleção'])} antes | "
      f"{formatar_pontos_percentuais(diferencas_depois['Taxa de seleção'])} depois")
for nome_genero in ["Feminino", "Masculino"]:
    falsos_positivos_antes = metricas_antes_mitigacao.by_group.loc[nome_genero, "Taxa de falsos positivos"]
    falsos_positivos_depois = metricas_depois_mitigacao.by_group.loc[nome_genero, "Taxa de falsos positivos"]
    print(f"Taxa de falsos positivos ({nome_genero}): {formatar_percentagem(falsos_positivos_antes)} antes | "
          f"{formatar_percentagem(falsos_positivos_depois)} depois")
print(f"Exatidão contra a aptidão real: {formatar_percentagem(exatidao_contra_aptidao)} antes | "
      f"{formatar_percentagem(exatidao_depois_mitigacao)} depois")

#### Limitações

- Os dados são __sintéticos__: o viés foi posto de propósito e os números não dizem nada sobre uma empresa real.
- Na vida real, a "aptidão real" não se conhece tão bem; sem ela, a equal opportunity é difícil de medir.
- O `ThresholdOptimizer` precisa do género no momento da decisão, o que pode ser problemático do ponto de vista legal e ético (por exemplo, no RGPD ou no AI Act). Alternativas como a mitigação durante o treino evitam isso.
- A mitigação não é perfeita no conjunto de teste (há uma parte aleatória e uma amostra finita) e pode ter custos noutras métricas.
- O custo desta mitigação são os falsos positivos das mulheres: corrigir uma métrica pode piorar outra.
- Não existe uma métrica universal de justiça: a escolha depende do contexto e dos trade-offs éticos.